In [1]:
import random
# import gymnasium as gym
import gym
# import gymnasium as gymnasium



import numpy as np
import tensorflow as tf
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
import rl
print(rl.__file__)
from rl.agents import DQNAgent
from rl.policy import BoltzmannQPolicy
from rl.memory import SequentialMemory

print("keras-rl2 import works ✅")


ModuleNotFoundError: No module named 'gym'

In [ ]:
import gym
import numpy as np

class KerasRLEnv(gym.Wrapper):
    def reset(self, **kwargs):
        result = self.env.reset(**kwargs)
        # Gym >=0.26 returns (obs, info)
        if isinstance(result, tuple):
            obs, info = result
        else:
            obs = result
        return np.array(obs, dtype=np.float32)

    def step(self, action):
        result = self.env.step(action)
        # Gym >=0.26 returns 5 values, older Gym returns 4
        if len(result) == 5:
            obs, reward, terminated, truncated, info = result
            done = terminated or truncated
        else:
            obs, reward, done, info = result
        return np.array(obs, dtype=np.float32), reward, done, info

env = KerasRLEnv(gym.make("CartPole-v1",render_mode="human"))


In [ ]:
# env = gym.make("CartPole-v1")
# env = gymnasium.make("CartPole-v1")
# env = gym.wrappers.TimeLimit(env, max_episode_steps=500)
states= env.observation_space.shape[0]
action= env.action_space.n
print(states)
print(action)

4
2


In [ ]:
episodes=10
for episode in range(1,episodes+1):
    state = env.reset()
    done =False
    score=0
    
    while not done:
        env.render()
        action = random.choice([0,1])
        obs, reward, done, info = env.step(action)
        score += reward
        
    
    print('Episode: {} Score: {}'.format(episode,score))


/home/usman/Documents/Robotics_AI/Some_projects/reinforcement_learning/rl_env/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/home/usman/Documents/Robotics_AI/Some_projects/reinforcement_learning/rl_env/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Episode: 1 Score: 19.0
Episode: 2 Score: 13.0
Episode: 3 Score: 17.0
Episode: 4 Score: 11.0
Episode: 5 Score: 12.0
Episode: 6 Score: 32.0
Episode: 7 Score: 11.0
Episode: 8 Score: 12.0
Episode: 9 Score: 28.0
Episode: 10 Score: 34.0


In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Flatten
from tensorflow.keras.optimizers import Adam


In [6]:
def build_models(states,actions):
    model =Sequential()
    model.add(Flatten(input_shape=(1,states)))
    model.add(Dense(24,activation='relu'))
    model.add(Dense(24,activation='relu'))
    model.add(Dense(actions,activation='linear'))
    return model 

In [7]:
model = build_models(states,action)


In [8]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 4)                 0         
                                                                 
 dense (Dense)               (None, 24)                120       
                                                                 
 dense_1 (Dense)             (None, 24)                600       
                                                                 
 dense_2 (Dense)             (None, 1)                 25        
                                                                 
Total params: 745 (2.91 KB)
Trainable params: 745 (2.91 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [9]:
from rl.agents import DQNAgent
from rl.policy import BoltzmannQPolicy
from rl.memory import SequentialMemory

In [10]:
def build_agents(model, actions):
    policy = BoltzmannQPolicy()
    memory = SequentialMemory(limit=50000, window_length=1)
    dqn = DQNAgent(model=model, memory=memory, policy=policy,
                   nb_actions=actions, nb_steps_warmup=10,
                   target_model_update=1e-2)
    return dqn

In [11]:
from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras.metrics import RootMeanSquaredError
dqn = build_agents(model, actions=action)
dqn.compile(Adam(learning_rate=1e-3), metrics=[RootMeanSquaredError()])
# dqn.fit(env, nb_steps=50000, visualize=False, verbose=1)

2025-09-24 01:05:35.245441: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2025-09-24 01:05:35.394696: W tensorflow/c/c_api.cc:305] Operation '{name:'dense_1_1/bias/Assign' id:130 op device:{requested: '', assigned: ''} def:{{{node dense_1_1/bias/Assign}} = AssignVariableOp[_has_manual_control_dependencies=true, dtype=DT_FLOAT, validate_shape=false](dense_1_1/bias, dense_1_1/bias/Initializer/zeros)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.


In [ ]:
dqn.fit(env, nb_steps=50000, visualize=False, verbose=1)


Training for 50000 steps ...
Interval 1 (0 steps performed)


/home/usman/Documents/Robotics_AI/Some_projects/reinforcement_learning/rl_env/lib/python3.10/site-packages/keras/src/engine/training_v1.py:2359: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,
2025-09-24 01:05:36.998690: W tensorflow/c/c_api.cc:305] Operation '{name:'dense_2/BiasAdd' id:75 op device:{requested: '', assigned: ''} def:{{{node dense_2/BiasAdd}} = BiasAdd[T=DT_FLOAT, _has_manual_control_dependencies=true, data_format="NHWC"](dense_2/MatMul, dense_2/BiasAdd/ReadVariableOp)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2025-09-24 01:05:37.135609: W tensorflow/c/c_api.cc:305] Operation '{name:'count_2/Assign' id:305 op device:{requested: '', assigned: ''} def:{{{node c

   11/10000 [..............................] - ETA: 4:24 - reward: 1.0000

/home/usman/Documents/Robotics_AI/Some_projects/reinforcement_learning/rl_env/lib/python3.10/site-packages/rl/memory.py:37: UserWarning: Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!
  warnings.warn('Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!')
2025-09-24 01:05:37.606037: W tensorflow/c/c_api.cc:305] Operation '{name:'dense_2_1/BiasAdd' id:159 op device:{requested: '', assigned: ''} def:{{{node dense_2_1/BiasAdd}} = BiasAdd[T=DT_FLOAT, _has_manual_control_dependencies=true, data_format="NHWC"](dense_2_1/MatMul, dense_2_1/BiasAdd/ReadVariableOp)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2025-09-24 01:05:38.621747: W tensorflow/c/c_api.cc:305] Operation '{name:'loss_3/AddN' id:409 op dev

  210/10000 [..............................] - ETA: 12:42 - reward: 1.0000done, took 16.678 seconds


: 

In [ ]:
# Evaluate trained agent visually
test_env = gym.make("CartPole-v1", render_mode="human")

obs, info = test_env.reset()
done = False
total_reward = 0

while not done:
    # keras-rl2 expects shape (1,1,states)
    action = dqn.forward(obs)  
    obs, reward, terminated, truncated, info = test_env.step(action)
    done = terminated or truncated
    total_reward += reward

print("Total reward in test episode:", total_reward)
test_env.close()


/home/usman/Documents/Robotics_AI/Some_projects/reinforcement_learning/rl_env/lib/python3.10/site-packages/keras/src/engine/training_v1.py:2359: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,
2025-09-18 16:04:41.201539: W tensorflow/c/c_api.cc:305] Operation '{name:'dense_2/BiasAdd' id:75 op device:{requested: '', assigned: ''} def:{{{node dense_2/BiasAdd}} = BiasAdd[T=DT_FLOAT, _has_manual_control_dependencies=true, data_format="NHWC"](dense_2/MatMul, dense_2/BiasAdd/ReadVariableOp)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2025-09-18 16:04:41.216736: W tensorflow/c/c_api.cc:305] Operation '{name:'total_3/Assign' id:310 op device:{requested: '', assigned: ''} def:{{{node t

Total reward in test episode: 10.0


In [ ]:
dqn.test(env,nb_episodes=15,visualize=False)

Testing for 15 episodes ...
Episode 1: reward: 10.000, steps: 10
Episode 2: reward: 9.000, steps: 9
Episode 3: reward: 9.000, steps: 9
Episode 4: reward: 9.000, steps: 9
Episode 5: reward: 10.000, steps: 10
Episode 6: reward: 9.000, steps: 9
Episode 7: reward: 9.000, steps: 9
Episode 8: reward: 9.000, steps: 9
Episode 9: reward: 9.000, steps: 9
Episode 10: reward: 9.000, steps: 9
Episode 11: reward: 9.000, steps: 9
Episode 12: reward: 8.000, steps: 8
Episode 13: reward: 10.000, steps: 10
Episode 14: reward: 9.000, steps: 9
Episode 15: reward: 9.000, steps: 9
